In [ ]:
import os
os.chdir('f:\\Projects\\GeneIndex')

In [ ]:
from PIL import Image
import pandas as pd
import numpy as np
import json

from tqdm import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
import pickle as pkl

import torch
import torch.nn as nn
from safetensors.torch import load_file
from torch.utils.tensorboard import SummaryWriter
from torchvision import tv_tensors
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection import FasterRCNN
from torchvision.ops import MultiScaleRoIAlign
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

from src.m0_ae.encoder import MAEEncoder
from src.m1_record_splitter.utils import read_labels, show_bbox, filter_rcnn_targets
from src.m1_record_splitter.dataset import ScanDataset

In [ ]:
IMG_W = 1920
IMG_H = 1600

D_MODEL = 384
ENCODER_PATH = Path('src/m0_ae/ae-v0.1.safetensors')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

In [ ]:
labs,imgs = read_labels(Path('src/m1_record_splitter/dataset/ds1_births'))
a,b = read_labels(Path('src/m1_record_splitter/dataset/ds3_al1'))
labs.extend(a)
imgs.extend(b)

imgs,labs = pd.Series(imgs), pd.Series(labs)

img_prefixes = imgs.map(lambda x:x.name).str.split('-').str[0]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=67)
split_idx_train, split_idx_val = next(gss.split(imgs, labs, img_prefixes))

train_imgs, train_labs = pd.Series(imgs)[split_idx_train], pd.Series(labs)[split_idx_train]
val_imgs, val_labs = pd.Series(imgs)[split_idx_val], pd.Series(labs)[split_idx_val]

In [ ]:
print('Subgroups in train:', train_imgs.map(lambda x:x.name).str.split('-').str[0].unique())
print('Subgroups in val:', val_imgs.map(lambda x:x.name).str.split('-').str[0].unique())

In [ ]:
train_ds = ScanDataset(train_imgs.reset_index(drop=True),train_labs.reset_index(drop=True))
val_ds = ScanDataset(val_imgs.reset_index(drop=True),val_labs.reset_index(drop=True))

In [ ]:
class LayerNorm2d(nn.Module):
    def __init__(self, num_channels, eps=1e-6):
        super().__init__()
        self.ln = nn.LayerNorm(num_channels, eps=eps)
    def forward(self, x):  # x: [N,C,H,W]
        x = x.permute(0, 2, 3, 1)
        x = self.ln(x)
        return x.permute(0, 3, 1, 2)

def out_conv_pair(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 1, bias=False), LayerNorm2d(out_ch),
        nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), LayerNorm2d(out_ch),
    )

class Backbone(nn.Module):
    def __init__(self, pretrained_encoder_path: Path|None=None):
        super().__init__()
        self.encoder = MAEEncoder(d_model=384)

        if pretrained_encoder_path:
            self.encoder.load_state_dict(load_file(pretrained_encoder_path))
            self.encoder.update_posec_interp()
            print('Loaded backbone encoder from:', pretrained_encoder_path)

        d = D_MODEL

        self.down = nn.Sequential(nn.MaxPool2d(2, 2), out_conv_pair(d, d))
        self.up1  = nn.Sequential(nn.Upsample(scale_factor=2), nn.Conv2d(d,d//2,3,1,1),
                                   out_conv_pair(d//2, d))
        self.up2  = nn.Sequential(
            nn.Upsample(scale_factor=2), nn.Conv2d(d,d//2,3,1,1), LayerNorm2d(d // 2), nn.GELU(),
            nn.Upsample(scale_factor=2), nn.Conv2d(d//2,d//4,3,1,1),
            out_conv_pair(d//4, d))
        self.feat_conv = out_conv_pair(d, d)

        self.out_channels = d

    def forward(self, x):
        feats = self.encoder(x,mask_perc=0) # [N, 384, x, y]

        down = self.down(feats)
        up1 = self.up1(feats)
        up2 = self.up2(feats)
        feats_conv = self.feat_conv(feats)

        return {
            '0': up2,
            '1': up1,
            '2': feats_conv,
            '3': down
        }

In [ ]:
class FakeTensorWithShape:
    def __init__(self, len, w,h):
        self.len = len
        self.h = h
        self.w = w

        self.shape = (self.h,self.w)

class FakeImgList:
    def __init__(self, len, h, w):
        """ This function replaces these calls:
            - image_list.tensors.shape[-2:]
            - image_list.image_sizes
        """

        self.len = len
        self.h = h
        self.w = w

        self.tensors = FakeTensorWithShape(self.len, self.w, self.h)
        self.image_sizes = [(self.h,self.w) for _ in range(self.len)]

In [ ]:
from collections import OrderedDict

class BBModel(nn.Module):
    def __init__(self):
        super(BBModel, self).__init__()
        
        anchor_generator = AnchorGenerator(
            sizes=(
                (32,),
                (64,),
                (128,),
                (256,),
            ),
            aspect_ratios=(
                (0.5, 1.0, 2.0),
                (0.5, 1.0, 2.0),
                (0.5, 1.0, 2.0),
                (0.5, 1.0, 2.0),
            ),
        )

        roi_pooler = MultiScaleRoIAlign(
            featmap_names=["0", "1", "2", "3"],
            output_size=7,
            sampling_ratio=2,
        )

        self.backbone = Backbone(pretrained_encoder_path=ENCODER_PATH)

        self.model = FasterRCNN(
            backbone=self.backbone,
            num_classes=3,
            rpn_anchor_generator=anchor_generator,
            box_roi_pool=roi_pooler,
            min_size=IMG_H,
            max_size=IMG_W
        )

    def skip_backbone(self, features, targets=None):
        b_size = len(targets)

        if self.training:
            if targets is None:
                torch._assert(False, "targets should not be none when in training mode")
            else:
                for target in targets:
                    boxes = target["boxes"]
                    if isinstance(boxes, torch.Tensor):
                        torch._assert(
                            len(boxes.shape) == 2 and boxes.shape[-1] == 4,
                            f"Expected target boxes to be a tensor of shape [N, 4], got {boxes.shape}.",
                        )
                    else:
                        torch._assert(
                            False,
                            f"Expected target boxes to be of type Tensor, got {type(boxes)}.",
                        )

        original_image_sizes: list[tuple[int, int]] = [(IMG_H, IMG_W) for _ in range(b_size)]

        images = FakeImgList(b_size, IMG_H, IMG_W)

        # features = self.backbone(images.tensors)

        if isinstance(features, torch.Tensor):
            features = OrderedDict([("0", features)])
        proposals, proposal_losses = self.model.rpn(images, features, targets)
        detections, detector_losses = self.model.roi_heads(features, proposals, images.image_sizes, targets)
        detections = self.model.transform.postprocess(
            detections, images.image_sizes, original_image_sizes
        )  # type: ignore[operator]

        losses = {}
        losses.update(detector_losses)
        losses.update(proposal_losses)

        if torch.jit.is_scripting():
            if not self.model._has_warned:
                print('RCNN always returns a (Losses, Detections) tuple in scripting')
                self.model._has_warned = True
            return losses, detections
        else:
            return self.model.eager_outputs(losses, detections)
        

    def forward(self, x,targ=None):
        y = self.model(x,targ)

        return y


In [ ]:
from torch.utils.data._utils.collate import default_collate
import h5py

class BackboneFeatureExtract(Dataset):
    def __init__(self, dataset, backbone: Backbone, feat_file_path: Path, labs_file_path: Path, override=False):
        self.feat_file_path = feat_file_path
        self.labs_file_path = labs_file_path
        self.labels = []

        self.len = len(dataset)

        with torch.no_grad():
            feats = backbone(dataset[0][0].unsqueeze(0).to(DEVICE))  
            self.all_keys = feats.keys()

        if not self.feat_file_path.exists() or override:
            with h5py.File(self.feat_file_path, "w") as f:
                dataset_dict = {}
                for k,v in feats.items():
                    d_model, feats_h, feats_w = v.size(1), v.size(2), v.size(3)
                    dataset_dict[k] = f.create_dataset(
                        k, 
                        shape=(self.len, d_model, feats_h, feats_w), 
                        dtype=np.float32,
                        chunks=(1, d_model, feats_h, feats_w)
                    )

                for z, (img, labs) in enumerate(tqdm(dataset)):
                    if z>15: break
                    self.labels.append(labs)

                    with torch.no_grad():
                        feats = backbone(img.unsqueeze(0).to(DEVICE))   
                    del img
                    feats = {k: v.cpu()[0] for k,v in feats.items()}

                    for k,v in feats.items():
                        dataset_dict[k][z] = v
            pkl.dump(self.labels, self.labs_file_path.open('wb'))
        else:
            self.labels = pkl.load(self.labs_file_path.open('rb'))

        self.file = h5py.File(self.feat_file_path, "r")
    def __getitem__(self, idx):
        return {k: torch.from_numpy(self.file[k][idx]) for k in self.all_keys}, self.labels[idx]
    def __len__(self):
        return len(self.labels)
    def __delete__(self, instance):
        self.file.close()
# val_ds_preloaded = BackbonePreload(val_ds, mdl.backbone, Path('src/m1_record_splitter/preloaded_imgs_val.h5'))

In [ ]:
def col_fn(b): 
    return tuple(zip(*b))
def feats_col_fn(b: list[tuple[dict,dict]]): 
   #  print(b)
   #  print({k: [sd[k] for sd in b] for k in b[0].keys()})
    return {k: torch.stack([sd[0][k] for sd in b]) for k in b[0][0].keys()}, [i[1] for i in b]

In [ ]:
aug = v2.Compose([
    # v2.RandomRotation(0),
    # v2.RandomPerspective(distortion_scale=0.3),
    # v2.RandomAffine(10,(0.05,0.05)),
    v2.ColorJitter(0,0,0),
    # v2.ClampBoundingBoxes(),
])

In [ ]:
torch.cuda.empty_cache()


mdl = BBModel().to(DEVICE)
mdl.backbone.encoder.requires_grad_(False)

train_ds_features = BackboneFeatureExtract(train_ds, mdl.backbone, Path('src/m1_record_splitter/features_imgs_train.h5'), Path('src/m1_record_splitter/features_imgs_labels_train.pkl'))
val_ds_features = BackboneFeatureExtract(val_ds, mdl.backbone, Path('src/m1_record_splitter/features_imgs_val.h5'), Path('src/m1_record_splitter/features_imgs_labels_val.pkl'))

In [ ]:
def loss_on_val(mdl, val_dl):
    mdl.train()
    val_loss_avg = 0
    with torch.no_grad():
        for img,targ in val_dl:
            img = [i.to(DEVICE) for i in img]
            targ = [{k:v.to(DEVICE) for k,v in i.items()} for i in targ]

            ret_dict = mdl(img,targ)

            val_loss_avg += sum([v for k,v in ret_dict.items()]).item()
    val_loss_avg /= len(val_dl)

    return val_loss_avg

def pred_ov_val(mdl, writer, targ, epoch):
    mdl.eval()
    with torch.no_grad():
        d = mdl(img)

        img = [i.cpu() for i in img]
        d = [{k:v.cpu() for k,v in i.items()} for i in d]
        
        writer.add_figure('Pred on val', show_bbox(img[0],d[0], targ[0], ret_fig=True,console=False),global_step=epoch,close=True)
    mdl.train()

In [ ]:
tb_writer = SummaryWriter('tensorboard_runs/m1_record_spliter/ae-v0.1/stage1/run-test')

In [ ]:
with torch.no_grad():
    torch.cuda.empty_cache()

torch.cuda.empty_cache()
torch.cuda.synchronize()
import gc
gc.collect()

In [ ]:
# STAGE 1 - FROZEN BACKBONE 
mdl.backbone.encoder.requires_grad_(False)

opt = torch.optim.AdamW(mdl.parameters(), lr=0.0002)
crit = nn.MSELoss()
train_features_dl = DataLoader(train_ds_features, batch_size=2, shuffle=True,collate_fn=feats_col_fn)
val_features_dl = DataLoader(val_ds_features, batch_size=2, shuffle=True,collate_fn=feats_col_fn)
sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt,len(train_features_dl),T_mult=2)

for epoch in range(501):
    mdl.train()
    
    train_loss_avg = 0
    t = tqdm(train_features_dl,desc=f'Epoch {epoch}')
    for feats,targ in t:
        feats = {k:v.to(DEVICE) for k,v in feats.items()}
        targ = [{k:v.to(DEVICE) for k,v in i.items()} for i in targ]
    
        # img,targ = aug(img,targ)

        ret_dict = mdl.skip_backbone(feats,targ)

        loss = sum([v for k,v in ret_dict.items()])
        t.set_postfix_str(f'Loss: {loss.item():.03f}, Lr: {sch.get_last_lr()}')
        
        train_loss_avg += loss.item()

        loss.backward()
        opt.step()
        opt.zero_grad()

        sch.step()
    train_loss_avg/= len(train_features_dl)

    # val_loss_avg = loss_on_val(mdl, val_features_dl)

    tb_writer.add_scalar('Train loss', train_loss_avg, epoch)
    # tb_writer.add_scalar('Val loss', val_loss_avg, epoch)
    tb_writer.add_scalar('Learning rate', sch.get_last_lr()[0], epoch)

In [ ]:

opt = torch.optim.AdamW(mdl.parameters(), lr=0.0002)

crit = nn.MSELoss()
train_dl = DataLoader(train_ds, batch_size=1, shuffle=True,collate_fn=col_fn)
val_dl = DataLoader(val_ds, batch_size=1, shuffle=True,collate_fn=col_fn)
sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt,len(train_dl),T_mult=2)

for epoch in range(501):
    mdl.train()
    
    train_loss_avg = 0
    t = tqdm(train_dl,desc=f'Epoch {epoch}')
    for img,targ in t:
        img = [i.to(DEVICE) for i in img]
        targ = [{k:v.to(DEVICE) for k,v in i.items()} for i in targ]
    
        img,targ = aug(img,targ)
        targ = filter_rcnn_targets(targ)

        ret_dict = mdl(img,targ)

        loss = sum([v for k,v in ret_dict.items()])
        t.set_postfix_str(f'Loss: {loss.item():.03f}, Lr: {sch.get_last_lr()}')
        
        train_loss_avg += loss.item()

        loss.backward()
        opt.step()
        opt.zero_grad()

        sch.step()
    train_loss_avg/= len(train_dl)

    val_loss_avg = loss_on_val(mdl, val_dl)
    pred_ov_val(mdl, tb_writer, targ, epoch)

    tb_writer.add_scalar('Train loss', train_loss_avg, epoch)
    tb_writer.add_scalar('Val loss', val_loss_avg, epoch)
    tb_writer.add_scalar('Learning rate', sch.get_last_lr()[0], epoch)

In [ ]:
# import pickle
# pickle.dump(mdl, open('ae_run2_no-aug_12l.pkl','wb'))